# house price prediction regression project

## Project Goal

## Load Modules / Libraries

In [2]:
# install below libraries by following command prompt
"""
pip install numpy
pip install pandas
pip install matplotlib
pip install seaborn
"""
# load libraries
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [16]:
data_path = r"C:\Users\Admin\Desktop\house_price_prediction\data.csv"
df_data=pd.read_csv(data_path)
print("shape of df_data: ",df_data.shape)

shape of df_data:  (7691, 10)


In [19]:
df_data.head()

,house_type,locality,city,area,beds,bathrooms,balconies,furnishing,area_rate,rent
0,"2 BHK Flat for Rent in Oberoi Woods, Goregaon ...",Goregaon East,Mumbai,897.0,2,2,0,Semi-Furnished,134.0,120000.0
1,"1 BHK Flat for Rent in Sapphire Lakeside, Powa...",Powai,Mumbai,490.0,1,1,0,Semi-Furnished,82.0,40000.0
2,1 BHK House for Rent in Mundhwa Pune,Mundhwa,Pune,550.0,1,1,0,Unfurnished,22.0,12000.0
3,"2 BHK Flat for Rent in Hingna, Nagpur",Hingna,Nagpur,1000.0,2,2,0,Unfurnished,8.0,8000.0
4,1 BHK Flat for Rent in Unique Star Harsh Vihar...,Mira Road,Mumbai,595.0,1,1,0,Unfurnished,25.0,15000.0


## Know your data

## Breif  info

In [23]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7691 entries, 0 to 7690
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   house_type  7691 non-null   object 
 1   locality    7691 non-null   object 
 2   city        7691 non-null   object 
 3   area        7691 non-null   float64
 4   beds        7691 non-null   int64  
 5   bathrooms   7691 non-null   int64  
 6   balconies   7691 non-null   int64  
 7   furnishing  7691 non-null   object 
 8   area_rate   7691 non-null   float64
 9   rent        7691 non-null   float64
dtypes: float64(3), int64(3), object(4)
memory usage: 601.0+ KB


In [24]:
print ("shape of integrated data/ DF:",df_data.shape)

shape of integrated data/ DF: (7691, 10)


In [24]:
print ("shape of integrated data/ DF:",df_data.shape)

shape of integrated data/ DF: (7691, 10)


In [24]:
print ("shape of integrated data/ DF:",df_data.shape)

shape of integrated data/ DF: (7691, 10)


In [24]:
print ("shape of integrated data/ DF:",df_data.shape)

shape of integrated data/ DF: (7691, 10)


In [7]:
# =========================================
# 1. Import Libraries
# =========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================================
# 2. Load Dataset
# =========================================
data_path = r"C:\Users\Admin\Desktop\house_price_prediction\data.csv"
df = pd.read_csv(data_path)

# =========================================
# 3. Data Inspection
# =========================================
print("Shape:", df.shape)

print("\nFirst 5 rows:\n", df.head())
print("\nLast 5 rows:\n", df.tail())

print("\nColumns:\n", df.columns)

print("\nInfo:")
df.info()

print("\nMissing Values:\n", df.isnull().sum())

# =========================================
# 4. Data Cleaning
# =========================================
# Remove duplicates
df = df.drop_duplicates()

# Handle missing values (FIXED)
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# =========================================
# 5. Feature Engineering
# =========================================
# Example feature
if "beds" in df.columns and "rent" in df.columns:
    df["price_per_room"] = df["rent"] / (df["beds"] + 1)

# =========================================
# 6. Outlier Removal (ONLY numeric)
# =========================================
def remove_outliers(df):
    numeric_df = df.select_dtypes(include=[np.number])

    Q1 = numeric_df.quantile(0.25)
    Q3 = numeric_df.quantile(0.75)
    IQR = Q3 - Q1

    mask = ~((numeric_df < (Q1 - 1.5 * IQR)) | 
             (numeric_df > (Q3 + 1.5 * IQR))).any(axis=1)

    return df[mask]

df = remove_outliers(df)

# =========================================
# 7. Encoding (Categorical → Numerical)
# =========================================
df = pd.get_dummies(df, drop_first=True)

# =========================================
# 8. Split Data
# =========================================
X = df.drop("rent", axis=1)
y = df["rent"]

# Feature Scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# 9. Model Training
# =========================================
lr = LinearRegression()
rf = RandomForestRegressor()

# =========================================
# 10. K-Fold Cross Validation
# =========================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr_score = cross_val_score(lr, X_train, y_train, cv=kf, scoring="neg_mean_squared_error")
rf_score = cross_val_score(rf, X_train, y_train, cv=kf, scoring="neg_mean_squared_error")

print("\nLinear Regression CV Score:", np.mean(lr_score))
print("Random Forest CV Score:", np.mean(rf_score))

# =========================================
# 11. GridSearchCV (Best Model)
# =========================================
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=5, scoring="neg_mean_squared_error")
grid.fit(X_train, y_train)

print("\nBest Parameters:", grid.best_params_)

best_model = grid.best_estimator_

# =========================================
# 12. Final Evaluation
# =========================================
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("\nFinal Model Performance:")
print("MAE:", mae)
print("MSE:", mse)

# =========================================
# 13. Save Model
# =========================================
import pickle

pickle.dump(best_model, open("model.pkl", "wb"))

print("\nModel saved successfully!")

Shape: (7691, 10)

First 5 rows:
                                           house_type       locality    city  \
0  2 BHK Flat for Rent in Oberoi Woods, Goregaon ...  Goregaon East  Mumbai   
1  1 BHK Flat for Rent in Sapphire Lakeside, Powa...          Powai  Mumbai   
2               1 BHK House for Rent in Mundhwa Pune        Mundhwa    Pune   
3              2 BHK Flat for Rent in Hingna, Nagpur         Hingna  Nagpur   
4  1 BHK Flat for Rent in Unique Star Harsh Vihar...      Mira Road  Mumbai   

     area  beds  bathrooms  balconies      furnishing  area_rate      rent  
0   897.0     2          2          0  Semi-Furnished      134.0  120000.0  
1   490.0     1          1          0  Semi-Furnished       82.0   40000.0  
2   550.0     1          1          0     Unfurnished       22.0   12000.0  
3  1000.0     2          2          0     Unfurnished        8.0    8000.0  
4   595.0     1          1          0     Unfurnished       25.0   15000.0  

Last 5 rows:
               

In [12]:
# =========================================
# 1. Import Libraries
# =========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================
# 2. Load Dataset
# =========================================
data_path = r"C:\Users\Admin\Desktop\house_price_prediction\data.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)

# =========================================
# 3. Check Data
# =========================================
print("\nFirst 5 rows:\n", df.head())
print("\nLast 5 rows:\n", df.tail())
print("\nColumns:\n", df.columns)
print("\nInfo:\n")
print(df.info())

# =========================================
# 4. Handle Missing Values
# =========================================
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# =========================================
# 5. Feature Engineering
# =========================================
# Example: total rooms
if "beds" in df.columns and "bathroom" in df.columns:
    df["total_rooms"] = df["beds"] + df["bathroom"]

# =========================================
# 6. Convert Categorical → Numerical
# =========================================
df = pd.get_dummies(df, drop_first=True)

# =========================================
# 7. Split Features & Target
# =========================================
X = df.drop("rent", axis=1)
y = df["rent"]

# =========================================
# 8. Save Columns BEFORE scaling ⚠️
# =========================================
columns = X.columns

# =========================================
# 9. Scaling
# =========================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

# =========================================
# 10. Train-Test Split
# =========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# 11. Model Training (GridSearchCV)
# =========================================
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10]
}

grid = GridSearchCV(
    RandomForestRegressor(),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)

# =========================================
# 12. Prediction
# =========================================
y_pred = best_model.predict(X_test)

# =========================================
# 13. Evaluation
# =========================================
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("\nMAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

# =========================================
# 14. Save Model (FINAL STEP)
# =========================================
import pickle

pickle.dump({
    "model": best_model,
    "scaler": scaler,
    "columns": columns
}, open("model.pkl", "wb"))

print("\nModel saved successfully! ✅")

Shape: (7691, 10)

First 5 rows:
                                           house_type       locality    city  \
0  2 BHK Flat for Rent in Oberoi Woods, Goregaon ...  Goregaon East  Mumbai   
1  1 BHK Flat for Rent in Sapphire Lakeside, Powa...          Powai  Mumbai   
2               1 BHK House for Rent in Mundhwa Pune        Mundhwa    Pune   
3              2 BHK Flat for Rent in Hingna, Nagpur         Hingna  Nagpur   
4  1 BHK Flat for Rent in Unique Star Harsh Vihar...      Mira Road  Mumbai   

     area  beds  bathrooms  balconies      furnishing  area_rate      rent  
0   897.0     2          2          0  Semi-Furnished      134.0  120000.0  
1   490.0     1          1          0  Semi-Furnished       82.0   40000.0  
2   550.0     1          1          0     Unfurnished       22.0   12000.0  
3  1000.0     2          2          0     Unfurnished        8.0    8000.0  
4   595.0     1          1          0     Unfurnished       25.0   15000.0  

Last 5 rows:
               

In [13]:
import pickle

pickle.dump({
    "model": best_model,
    "scaler": scaler,
    "columns": columns
}, open("model.pkl", "wb"))

print("Model saved successfully!")

Model saved successfully!


In [2]:
df = pd.read_csv("data.csv")

# Select columns
df = df[["location", "area", "bedrooms", "bathrooms", "balconies", "rent", "price"]]

# Drop nulls
df = df.dropna()

# -------- LOCATION CLEANING --------
df["location"] = df["location"].str.lower().str.strip()

location_counts = df["location"].value_counts()
valid_locations = location_counts[location_counts > 10].index

df["location"] = df["location"].apply(lambda x: x if x in valid_locations else "other")

# -------- AREA CLEANING --------
df["area"] = df["area"].astype(str)

df["area"] = df["area"].str.replace("sqft", "")
df["area"] = df["area"].str.replace("sq.ft", "")
df["area"] = df["area"].str.replace("sqm", "")
df["area"] = df["area"].str.strip()

df["area"] = pd.to_numeric(df["area"], errors="coerce")

# Drop invalid area rows
df = df.dropna()

# -------- FEATURES --------
X = df[["location", "area", "bedrooms", "bathrooms", "balconies"]]
y_rent = df["rent"]
y_price = df["price"]

# One-hot encoding
X_encoded = pd.get_dummies(X, columns=["location"])

columns = X_encoded.columns

KeyError: "['location', 'bedrooms', 'price'] not in index"

In [5]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# -------------------------------
# 1. LOAD DATA
# -------------------------------
df = pd.read_csv("data.csv")

# -------------------------------
# 2. CLEAN COLUMN NAMES
# -------------------------------
df.columns = df.columns.str.strip().str.lower()

print("Columns:", df.columns)

# -------------------------------
# 3. RENAME COLUMNS
# -------------------------------
df = df.rename(columns={
    "locality": "location",
    "beds": "bedrooms"
})

# -------------------------------
# 4. SELECT REQUIRED COLUMNS
# -------------------------------
df = df[["location", "area", "bedrooms", "bathrooms", "balconies", "rent"]]

# Drop null values
df = df.dropna()

# -------------------------------
# 5. CLEAN LOCATION
# -------------------------------
df["location"] = df["location"].astype(str)
df["location"] = df["location"].str.lower().str.strip()

# Reduce rare locations
location_counts = df["location"].value_counts()
valid_locations = location_counts[location_counts > 5].index

df["location"] = df["location"].apply(
    lambda x: x if x in valid_locations else "other"
)

# -------------------------------
# 6. CLEAN AREA (convert to numeric)
# -------------------------------
df["area"] = df["area"].astype(str)

df["area"] = df["area"].str.replace("sqft", "", regex=False)
df["area"] = df["area"].str.replace("sq.ft", "", regex=False)
df["area"] = df["area"].str.replace("sqm", "", regex=False)

df["area"] = df["area"].str.strip()

df["area"] = pd.to_numeric(df["area"], errors="coerce")

# Drop invalid rows
df = df.dropna()

# -------------------------------
# 7. FEATURES & TARGET
# -------------------------------
X = df[["location", "area", "bedrooms", "bathrooms", "balconies"]]
y = df["rent"]

# -------------------------------
# 8. ONE HOT ENCODING
# -------------------------------
X_encoded = pd.get_dummies(X, columns=["location"])

columns = X_encoded.columns

# -------------------------------
# 9. TRAIN MODEL
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

# -------------------------------
# 10. EVALUATE MODEL
# -------------------------------
y_pred = model.predict(X_test)

print("\nModel Results:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

# -------------------------------
# 11. SAVE MODEL
# -------------------------------
with open("rent_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("columns.pkl", "wb") as f:
    pickle.dump(columns, f)

print("\n✅ Model saved successfully!")

Columns: Index(['house_type', 'locality', 'city', 'area', 'beds', 'bathrooms',
       'balconies', 'furnishing', 'area_rate', 'rent'],
      dtype='object')

Model Results:
MAE: 29110.86791252838
R2 Score: 0.539710188190339

✅ Model saved successfully!


In [7]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# -------------------------------
# 1. LOAD DATA
# -------------------------------
df = pd.read_csv("data.csv")

# -------------------------------
# 2. CLEAN COLUMN NAMES
# -------------------------------
df.columns = df.columns.str.strip().str.lower()
print("Columns:", df.columns)

# -------------------------------
# 3. RENAME COLUMNS
# -------------------------------
df = df.rename(columns={
    "locality": "location",
    "beds": "bedrooms"
})

# -------------------------------
# 4. CLEAN AREA (convert to numeric sqft)
# -------------------------------
df["area"] = df["area"].astype(str)

df["area"] = df["area"].str.replace("sqft", "", regex=False)
df["area"] = df["area"].str.replace("sq.ft", "", regex=False)
df["area"] = df["area"].str.replace("sqm", "", regex=False)

df["area"] = df["area"].str.strip()
df["area"] = pd.to_numeric(df["area"], errors="coerce")

# -------------------------------
# 5. CREATE PRICE (IMPORTANT 🔥)
# -------------------------------
df["area_rate"] = pd.to_numeric(df["area_rate"], errors="coerce")
df["price"] = df["rent"] * 250

# -------------------------------
# 6. SELECT REQUIRED COLUMNS
# -------------------------------
df = df[["location", "area", "bedrooms", "bathrooms", "balconies", "rent", "price"]]

df = df.dropna()

# -------------------------------
# 7. CLEAN LOCATION
# -------------------------------
df["location"] = df["location"].astype(str)
df["location"] = df["location"].str.lower().str.strip()

location_counts = df["location"].value_counts()
valid_locations = location_counts[location_counts > 5].index

df["location"] = df["location"].apply(
    lambda x: x if x in valid_locations else "other"
)

# -------------------------------
# 🔍 DEBUG PRINTS (VERY IMPORTANT)
# -------------------------------
print("\n--- CLEANED DATA ---")
print(df.head())

print("\n--- DATA TYPES ---")
print(df.dtypes)

# -------------------------------
# 8. FEATURES & TARGET
# -------------------------------
X = df[["location", "area", "bedrooms", "bathrooms", "balconies"]]
y_rent = df["rent"]
y_price = df["price"]

# -------------------------------
# 9. ENCODING
# -------------------------------
X_encoded = pd.get_dummies(X, columns=["location"])

columns = X_encoded.columns

print("\n--- ENCODED DATA ---")
print(X_encoded.head())

# -------------------------------
# 10. TRAIN RENT MODEL
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_rent, test_size=0.2, random_state=42
)

rent_model = LinearRegression()
rent_model.fit(X_train, y_train)

rent_pred = rent_model.predict(X_test)

print("\nRent Model:")
print("MAE:", mean_absolute_error(y_test, rent_pred))
print("R2:", r2_score(y_test, rent_pred))

# -------------------------------
# 11. TRAIN PRICE MODEL
# -------------------------------
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_encoded, y_price, test_size=0.2, random_state=42
)

price_model = LinearRegression()
price_model.fit(X_train_p, y_train_p)

price_pred = price_model.predict(X_test_p)

print("\nPrice Model:")
print("MAE:", mean_absolute_error(y_test_p, price_pred))
print("R2:", r2_score(y_test_p, price_pred))

# -------------------------------
# 12. SAVE MODELS
# -------------------------------
with open("rent_model.pkl", "wb") as f:
    pickle.dump(rent_model, f)

with open("price_model.pkl", "wb") as f:
    pickle.dump(price_model, f)

with open("columns.pkl", "wb") as f:
    pickle.dump(columns, f)

print("\n✅ Models saved successfully!")

Columns: Index(['house_type', 'locality', 'city', 'area', 'beds', 'bathrooms',
       'balconies', 'furnishing', 'area_rate', 'rent'],
      dtype='object')

--- CLEANED DATA ---
        location    area  bedrooms  bathrooms  balconies      rent       price
0  goregaon east   897.0         2          2          0  120000.0  30000000.0
1          powai   490.0         1          1          0   40000.0  10000000.0
2        mundhwa   550.0         1          1          0   12000.0   3000000.0
3          other  1000.0         2          2          0    8000.0   2000000.0
4      mira road   595.0         1          1          0   15000.0   3750000.0

--- DATA TYPES ---
location      object
area         float64
bedrooms       int64
bathrooms      int64
balconies      int64
rent         float64
price        float64
dtype: object

--- ENCODED DATA ---
     area  bedrooms  bathrooms  balconies  location_abbigere  \
0   897.0         2          2          0              False   
1   490.0       